## Merging _pandas_ DataFrames

In [ ]:
import pandas as pd

In [ ]:
unemployment = pd.read_csv('../data/tn_unemployment.csv')
unemployment.head(2)

This unemployment data was obtained from the Burea of Labor Statistics.

Now, let's bring in our second DataFrame.

In [ ]:
population = pd.read_csv('../data/tn_population.csv')
population.head(2)

Our goal is to combine the unemployment and population data. In order to do this, _pandas_ needs a common column to join on. 

Notice that the `Name` column from `unemployment` and the `County Name` column from `population` almost work, except that `Name` also includes ", TN". Let's remove this part so that we have a column to join on.

### Brief Detour - String Methods

When working with text data in `pandas`, it is often useful to utilize the built-in sting methods. To use these methods, you must prepend a `.str` before the desired method.

For example, we can make column entirely uppercase using the `upper()` method.

In [ ]:
population['County Name'].str.upper()

Another often useful method is the `replace()` method. To use this method, specify what pattern you want to replace and then the replacement text.

In [ ]:
unemployment['Period'].str.replace('21', '2021')

A method that we can use that will allow us to merge our dataframes is the `.str.split()` method.

Notice that if we split on the comma, the first piece will match what is contained in the `County Name` column.

In [ ]:
unemployment['Name'].str.split(',')

By default, this method returns a list. We can make it return a DataFrame by using the `expand` argument.

In [ ]:
unemployment['Name'].str.split(',', expand = True)

We only want the first column. Notice that we are using 0 as an integer and not as a string to access this column.

In [ ]:
unemployment['Name'].str.split(',', expand = True)[0]

Finally, we can assign this back to the `Name` column.

In [ ]:
unemployment['Name'] = unemployment['Name'].str.split(',', expand = True)[0]

In [ ]:
unemployment.head()

Now, we are ready to merge.

In [ ]:
population.head(1)

In [ ]:
unemployment.head(1)

**Option 1:** Use the `left_on` and `right_on` arguments to tell pandas what to merge on.

In [ ]:
pd.merge(left = population, 
         right = unemployment, 
         left_on = 'County Name', 
         right_on = 'Name')

Notice that we end up with a duplicated column (with different names).

**Option 2:** Rename the merging column for one of our DataFrames.

In [ ]:
pd.merge(left = population,
         right = unemployment.rename(columns = {'Name': 'County Name'}))

We can even select out just the columns that we need after the merge:

In [ ]:
pd.merge(left = population,
         right = unemployment[['Name', 'unemployment_rate']].rename(columns = {'Name': 'County Name'}))

Once we're happy with the results, we can save them to a DataFrame.

In [ ]:
tn_counties = pd.merge(
    left = population,
    right = unemployment[['Name', 'unemployment_rate']].rename(columns = {'Name': 'County Name'})
)

## Cardinalities and Validating your Merge

With related tables, there are 3 types of relationships that can occur:

* **One to One**: Each row from the first table matches with exactly one row from the second. This is the type of merge we performed with the unemployment data and the population data, where each county appeared in each table exactly once.
* **Many to One:** Each row from the first table matches one or more rows from the second. This occurs, for example, if you have a table with customer information and want to merge it with orders data. Each customer could have multiple orders.
* **Many to Many:** Each row in the first table can match with multiple rows in the second, and each row in the second can match with multiple rows in the first. For example, if we have a table containing students and a table containing classes, we might have a many-to-many relationship, where each student can be enrolled in multiple classes and each class can have multiple students enrolled.


The most common type of relationship would be a many-to-one or one-to-one, with many-to-many occurring infrequently. It is often a good idea to validate that you are getting the correct type of merge to detect if there are any data issues. 

The pandas merge function allows us to check this using the `validate` argument. For example, if we wanted to verify that our unemployment/population merge was as expected, we could do as follows. If it doesn't, we will get a MergeError.

In [ ]:
pd.merge(left = population, 
         right = unemployment, 
         left_on = 'County Name', 
         right_on = 'Name',
         validate = "one_to_one")

Something else we might want to validate is that all rows had a match. We might want to make our merge an outer merge to ensure that we don't lose any rows. We'll also set the indicator argument to True.

In [ ]:
indicator_merge = pd.merge(
    left = population, 
    right = unemployment, 
    left_on = 'County Name', 
    right_on = 'Name',
    validate = "one_to_one",
    how = 'outer',
    indicator = True
)
indicator_merge.head(2)

The _merge column will indicate whether a row came from just the left table, just the right table, or from both.

In [ ]:
indicator_merge[indicator_merge['_merge'] != 'both']

When writing code that you want to reuse, you can even add an `assert` statement. For these, you add a condition you want to check. If this condition is True, the assert statement does not return anything. However, if it is False, it will produce an error.

Assert statements can be useful especially when writing code that you would like to reuse as they can check assumptions and flag issues about the data that you are working with, even if that data changes in the future.

In [ ]:
assert len(indicator_merge[indicator_merge['_merge'] != 'both']) == 0, "Some rows didn't have a match."

In this case, all resulting rows had matches in both tables so this did not produce an error.

### Additional Examples: Olympics Data

Let's look at some other examples using some datasets from the summer and winter olympics.

In [ ]:
athletes = pd.read_csv("../data/athletes.csv")
summer_games = pd.read_csv("../data/summer_games.csv")
winter_games = pd.read_csv("../data/winter_games.csv")
countries = pd.read_csv("../data/countries.csv")

In [ ]:
countries.head(2)

In [ ]:
summer_games.head(2)

The summer_games table contains rows for each event and each athlete. If I merge in the countries information, I would expect many rows to match with a given country. Let's see what happens if I use the one_to_one check.

In [ ]:
pd.merge(
    left=countries,
    right=summer_games,
    left_on="id",
    right_on="country_id",
    validate="one_to_one"
)

On the other hand, if I do a one_to_many check, I don't get an error.

In [ ]:
pd.merge(
    left=countries,
    right=summer_games,
    left_on="id",
    right_on="country_id",
    validate="one_to_many"
)

Similarly, if I merged the athletes table to the summer_games table, I would expect a one-to-many merge. Let's try it.

In [ ]:
pd.merge(
    left=athletes,
    right=summer_games,
    left_on="id",
    right_on="athlete_id",
    validate="one_to_many"
)

Let's investigate.

In [ ]:
duplicates = athletes['id'].value_counts()[lambda x: x > 1].index
duplicates

In [ ]:
athletes[athletes['id'].isin(duplicates)]

It looks like a problem with our data; one of the athletes, [Eva Vrobcov-Nvltov](https://en.wikipedia.org/wiki/Eva_Vrabcov%C3%A1-N%C3%BDvltov%C3%A1) is listed twice, once for the summer games and once for the winter games.

Finally, if I wanted to do some analysis on medal counts across countries, I might be tempted to merge the summer and winter games tables on country id. However, notice this blows up the number of rows.

In [ ]:
print(f'Number of rows in the summer games table: {summer_games.shape[0]}')
print(f'Number of rows in the winter games table: {winter_games.shape[0]}')

In [ ]:
pd.merge(
    left=summer_games,
    right=winter_games,
    on='country_id'
)

I would probably be better off aggregating each table at the country level first before combining the results together.

Finally, let's merge the country data in with the winter games data. Let's say that I want to keep all countries, whether or not they have a match. This mean that I want to use a **right join**.

In [ ]:
wg_countries = pd.merge(
    left=winter_games,
    right=countries,
    left_on="country_id",
    right_on="id",
    how="right",
    validate="many_to_one",
    indicator=True
)

Let's see what happens if I check that all rows had a match using an assert statement.

In [ ]:
assert len(wg_countries[wg_countries['_merge'] != 'both']) == 0, "Some rows in the right table didn't have a match."

Let's see which countries didn't have a match.

In [ ]:
wg_countries[wg_countries['_merge'] != 'both']

It looks like there are 125 countries that did not appear in the winter games table.